# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [6]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
# 1.3. Drop any rows containing missing values (low proportion of nulls overall)
spaceship = spaceship.dropna(axis=0, how="any")

# 1.4. Extract only the “Deck” letter from Cabin (e.g., "B/0/P" → "B")
spaceship["Deck"] = spaceship["Cabin"].str.split("/", expand=True)[0]

# 1.5. Drop columns that are identifiers or no longer needed
spaceship = spaceship.drop(columns=["PassengerId", "Name", "Cabin"])

# 1.6. One‐hot encode all remaining categorical features except the target
categorical_cols = spaceship.select_dtypes(include=["object", "bool"]).columns.tolist()
categorical_cols.remove("Transported")  # leave target out of encoding

spaceship = pd.get_dummies(
    spaceship,
    columns=categorical_cols,
    drop_first=True
)

# 1.7. Separate features (X) and target (y)
y = spaceship["Transported"].astype(int)  # Convert boolean to 0/1
X = spaceship.drop(columns=["Transported"])

# 1.8. Split into train and test (80% train, 20% test, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 1.9. Feature scaling: standardize all numeric columns (KNN and many ensemble methods benefit)
#      Note: One‐hot columns are already 0/1, so scaling them isn’t strictly necessary, 
#      but we’ll scale all continuous columns for consistency.

# Identify numeric columns (dtype float or int)
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Preprocessing complete.")
print(f"  - X_train shape: {X_train.shape}")
print(f"  - X_test  shape: {X_test.shape}")
print(f"  - y_train distribution: {np.bincount(y_train)}")
print(f"  - y_test  distribution: {np.bincount(y_test)}")


Preprocessing complete.
  - X_train shape: (5284, 19)
  - X_test  shape: (1322, 19)
  - y_train distribution: [2623 2661]
  - y_test  distribution: [656 666]


**Perform Train Test Split**

In [9]:
# Just to confirm we have no missing values and everything is numeric:

print("Any nulls in X_train? ", X_train.isnull().sum().sum())
print("Any nulls in X_test?  ", X_test.isnull().sum().sum())

print("Dtypes in X_train:")
print(X_train.dtypes.value_counts())


Any nulls in X_train?  0
Any nulls in X_test?   0
Dtypes in X_train:
bool       13
float64     6
Name: count, dtype: int64


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [11]:
import pandas as pd  # pandas for data manipulation :contentReference[oaicite:9]{index=9}
import numpy as np   # numpy for numerical operations :contentReference[oaicite:10]{index=10}

from sklearn.model_selection import train_test_split  # split data :contentReference[oaicite:11]{index=11}
from sklearn.preprocessing import StandardScaler  # standardize numeric features :contentReference[oaicite:12]{index=12}

# 1. Load and drop nulls
spaceship = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
)  # load remote CSV :contentReference[oaicite:13]{index=13}
spaceship = spaceship.dropna(axis=0, how="any")  # drop any row with NaN :contentReference[oaicite:14]{index=14}

# 2. Extract 'Deck' from 'Cabin' (format: Deck/Room/Side)
spaceship["Deck"] = spaceship["Cabin"].str.split("/", expand=True)[0]  # take string before '/' :contentReference[oaicite:15]{index=15}

# 3. Drop identifier columns
spaceship = spaceship.drop(columns=["PassengerId", "Name", "Cabin"])  # remove ID and raw Cabin :contentReference[oaicite:16]{index=16}

# 4. One-hot encode all categorical features except target
cat_cols = spaceship.select_dtypes(include=["object", "bool"]).columns.tolist()
cat_cols.remove("Transported")  # target is boolean, so leave it for y :contentReference[oaicite:17]{index=17}

spaceship = pd.get_dummies(
    spaceship,
    columns=cat_cols,
    drop_first=True  # avoid dummy variable trap :contentReference[oaicite:18]{index=18}
)

# 5. Separate features (X) and target (y)
y = spaceship["Transported"].astype(int)  # convert boolean to 0/1 :contentReference[oaicite:19]{index=19}
X = spaceship.drop(columns=["Transported"])

# 6. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y  # keep class proportions :contentReference[oaicite:20]{index=20}
)

# 7. Scale numeric features
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()  # identify numeric dtypes :contentReference[oaicite:21]{index=21}

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])


In [12]:
from sklearn.neighbors import KNeighborsClassifier  # KNN for base :contentReference[oaicite:22]{index=22}
from sklearn.ensemble import BaggingClassifier  # ensemble Bagging (pasting if bootstrap=False) :contentReference[oaicite:23]{index=23}
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # evaluation metrics :contentReference[oaicite:24]{index=24}

# Instantiate base KNN
knn_base = KNeighborsClassifier(n_neighbors=5)  # default k=5 :contentReference[oaicite:25]{index=25}

# Corrected: use 'estimator' instead of deprecated 'base_estimator'
bagging = BaggingClassifier(
    estimator=knn_base,
    n_estimators=50,
    max_samples=1.0,    # draw 100% of training samples for each base learner (pasting)
    max_features=1.0,
    bootstrap=False,    # no replacement → pasting :contentReference[oaicite:26]{index=26}
    random_state=42
)
bagging.fit(X_train, y_train)
y_pred_bag = bagging.predict(X_test)

acc_bag = accuracy_score(y_test, y_pred_bag)
print("Bagging (Pasting) Accuracy:", acc_bag)
print(classification_report(y_test, y_pred_bag))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bag))


c:\Users\michele\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\michele\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\michele\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\michele\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\michele\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi

Bagging (Pasting) Accuracy: 0.7639939485627837
              precision    recall  f1-score   support

           0       0.76      0.76      0.76       656
           1       0.76      0.77      0.77       666

    accuracy                           0.76      1322
   macro avg       0.76      0.76      0.76      1322
weighted avg       0.76      0.76      0.76      1322

Confusion Matrix:
 [[497 159]
 [153 513]]


- Random Forests

In [13]:
from sklearn.ensemble import RandomForestClassifier  # RF classifier :contentReference[oaicite:29]{index=29}

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", acc_rf)
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


Random Forest Accuracy: 0.7753403933434191
              precision    recall  f1-score   support

           0       0.77      0.78      0.77       656
           1       0.78      0.77      0.78       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

Confusion Matrix:
 [[511 145]
 [152 514]]


- Gradient Boosting

In [14]:
from sklearn.ensemble import GradientBoostingClassifier  # GBM :contentReference[oaicite:32]{index=32}

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

acc_gb = accuracy_score(y_test, y_pred_gb)
print("Gradient Boosting Accuracy:", acc_gb)
print(classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))


Gradient Boosting Accuracy: 0.783661119515885
              precision    recall  f1-score   support

           0       0.82      0.72      0.77       656
           1       0.75      0.85      0.80       666

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322

Confusion Matrix:
 [[471 185]
 [101 565]]


- Adaptive Boosting

In [16]:
from sklearn.ensemble import AdaBoostClassifier  # AdaBoost :contentReference[oaicite:35]{index=35}

# If you want a custom base learner (e.g., deeper trees), you can do:
# from sklearn.tree import DecisionTreeClassifier
# ab = AdaBoostClassifier(
#     estimator=DecisionTreeClassifier(max_depth=2),
#     n_estimators=50,
#     learning_rate=1.0,
#     random_state=42
# )

# But by default, AdaBoostClassifier uses DecisionTreeClassifier(max_depth=1) stumps
ab = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)  # no need for base_estimator—defaults to 1‐level decision tree :contentReference[oaicite:36]{index=36}

ab.fit(X_train, y_train)
y_pred_ab = ab.predict(X_test)

acc_ab = accuracy_score(y_test, y_pred_ab)
print("AdaBoost Accuracy:", acc_ab)
print(classification_report(y_test, y_pred_ab))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ab))


AdaBoost Accuracy: 0.7874432677760969
              precision    recall  f1-score   support

           0       0.81      0.74      0.78       656
           1       0.77      0.83      0.80       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Confusion Matrix:
 [[488 168]
 [113 553]]


c:\Users\michele\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Which model is the best and why?

In [ ]:
### Bagging (Pasting) Accuracy:     0.76xx
## Random Forest Accuracy:         0.78xx
## Gradient Boosting Accuracy:     0.78xx
## AdaBoost Accuracy:              0.79xx
